# 15 — Data Skipping, Predicate Pushdown & Query Optimization

**What you will learn:**
- `explain()` — reading Spark query plans
- Predicate pushdown — filters pushed into the data source
- Column pruning — reading only required columns
- Partition pruning — skipping entire folders
- Delta Lake data skipping with file-level statistics
- Z-ORDER — co-locate related data for better skipping
- Bloom filters on Delta tables
- `ANALYZE TABLE` — collecting statistics
- Common anti-patterns that kill performance

## 1. Reading Query Plans with explain()

The query plan is Spark's execution blueprint.
Always check the plan when debugging performance.

| Mode | Shows |
|---|---|
| `explain()` | Simple physical plan |
| `explain(True)` | Full plan with all phases |
| `explain(mode="simple")` | Short physical plan |
| `explain(mode="formatted")` | Readable structured plan (recommended) |
| `explain(mode="extended")` | Parsed → Analyzed → Optimized → Physical |
| `explain(mode="cost")` | With row count estimates |

In [ ]:
import pyspark.sql.functions as F

df = (spark.range(1, 1000001).toDF("id")
      .withColumn("dept", (F.col("id") % 4).cast("string"))
      .withColumn("salary", (F.col("id") * 137 % 70000 + 50000).cast("double")))

df.write.format("delta").mode("overwrite").save("/tmp/opt/skip_demo/")
df_delta = spark.read.format("delta").load("/tmp/opt/skip_demo/")

df_delta.filter(F.col("dept") == "1").select("id","salary")         .explain(mode="formatted")

## 2. Predicate Pushdown

Instead of reading all rows into Spark then filtering, Spark pushes the filter **into the data source** so only matching rows are returned.

Works with: Parquet, Delta, ORC, JDBC (as WHERE clause), Avro.

Look for **`PushedFilters`** in the explain output.

In [ ]:
df.write.mode("overwrite").parquet("/tmp/opt/emp_parquet/")
df_pq = spark.read.parquet("/tmp/opt/emp_parquet/")

# Filter pushed into Parquet reader — rows skipped at file level
df_pq.filter(F.col("dept") == "2").explain(mode="formatted")
# Look for: PushedFilters: [IsNotNull(dept), EqualTo(dept,2)]

In [ ]:
# Predicate pushdown BREAKS if you transform the column before filtering

# GOOD — filter on raw column, pushdown works
df_pq.filter(F.col("dept") == "2").explain(mode="simple")

# BAD — filter on derived column, Spark must read ALL rows first then filter
df_pq.withColumn("dept_int", F.col("dept").cast("int"))      .filter(F.col("dept_int") == 2).explain(mode="simple")
# Notice: no PushedFilters — full table scan

## 3. Column Pruning

Spark reads **only the columns you select** from columnar formats (Parquet, Delta, ORC).
This is automatic — but only works if you `select()` early, before reading unneeded columns.

Look for **`ReadSchema`** in the explain plan — it should list only selected columns.

In [ ]:
# Full scan — ReadSchema includes all columns
df_pq.explain(mode="formatted")

# Column-pruned — ReadSchema: struct<id:bigint, salary:double>
df_pq.select("id","salary").explain(mode="formatted")
# Only 2 columns read from disk instead of 3 — less I/O

## 4. Partition Pruning

When data is written with `partitionBy`, filtering on the partition column lets Spark skip entire folders.
This is the most impactful optimization for large data lakes.

In [ ]:
df.write.format("delta").mode("overwrite")   .partitionBy("dept")   .save("/tmp/opt/emp_partitioned/")

df_part = spark.read.format("delta").load("/tmp/opt/emp_partitioned/")

# Only the dept=1 folder is scanned — ~25% of total files
df_part.filter(F.col("dept") == "1").explain(mode="formatted")
# Look for: PartitionFilters: [isnotnull(dept#x), (dept#x = 1)]

## 5. Delta Lake Data Skipping

Delta Lake stores **min/max statistics** for every column in every data file (written to the transaction log).
When you filter on a column, Delta reads the log and **skips files** whose min/max range cannot contain the filtered value.

This works on **non-partitioned columns** — much finer-grained than partition pruning.

In [ ]:
df.write.format("delta").mode("overwrite").save("/tmp/opt/emp_delta_skip/")
df_ds = spark.read.format("delta").load("/tmp/opt/emp_delta_skip/")

# Delta skips files where max(id) < 900000
df_ds.filter(F.col("id") > 900000).explain(mode="formatted")
# Look for: delta_scan with pushedFilters showing id > 900000

from delta.tables import DeltaTable
DeltaTable.forPath(spark, "/tmp/opt/emp_delta_skip/")     .detail().select("numFiles","sizeInBytes").show()

## 6. Z-ORDER — Co-locate Related Data

Data skipping works best when related values are **physically co-located** in the same files.
`ZORDER BY` rewrites files so rows with similar column values land in the same file.

**Use Z-ORDER on columns you:**
- Filter on frequently
- That are NOT the partition column
- That have high cardinality (IDs, timestamps, customer codes)

In [ ]:
spark.sql("OPTIMIZE delta.`/tmp/opt/emp_delta_skip/` ZORDER BY (id, salary)")

# After Z-ORDER, queries on id + salary skip far more files
df_z = spark.read.format("delta").load("/tmp/opt/emp_delta_skip/")
df_z.filter((F.col("id") > 900000) & (F.col("salary") > 100000))     .explain(mode="formatted")

# Z-ORDER guidelines:
# Max 2-3 columns per ZORDER (effectiveness decreases beyond that)
# Only run after bulk writes — not on every small append
# Re-run periodically as data accumulates (or use Liquid Clustering in DBR 13.3+)

## 7. Bloom Filters on Delta Tables

Bloom filters are probabilistic data structures that answer:
**"Is value X definitely NOT in this file?"**
If yes, Spark skips the file entirely.

Best for: **high-cardinality equality filters** — e.g., `WHERE customer_id = 'ABC123'`

In [ ]:
spark.sql('''
    ALTER TABLE delta.`/tmp/opt/emp_delta_skip/`
    SET TBLPROPERTIES (
        "delta.dataSkippingNumIndexedCols" = "32",
        "delta.bloomFilter.id.enabled"     = "true",
        "delta.bloomFilter.id.numItems"    = "1000000",
        "delta.bloomFilter.id.fpp"         = "0.1"
    )
''')
# fpp = false positive probability (0.1 = 10% false positives)
# Bloom filters are written into files on the next OPTIMIZE run
spark.sql("OPTIMIZE delta.`/tmp/opt/emp_delta_skip/`")
print("Bloom filters enabled and materialized.")

## 8. ANALYZE TABLE — Help the Optimizer

Collects table-level and column-level statistics that Catalyst uses for better join ordering and size estimation.

In [ ]:
spark.sql("ANALYZE TABLE default.employees COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE default.employees COMPUTE STATISTICS FOR COLUMNS department, salary")

spark.sql("DESCRIBE EXTENDED default.employees")      .filter(F.col("col_name").isin("Statistics"))      .show(truncate=False)

## 9. Common Anti-Patterns That Kill Performance

In [ ]:
# BAD 1: collect() on large DataFrame → driver OOM
# df.collect()

# BAD 2: UDF for something a built-in handles
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
bad_udf = udf(lambda s: s.upper(), StringType())
# Use F.upper(col("name")) instead — 10x faster

# BAD 3: Filter AFTER expensive join
# df.join(big_table, "id").filter("status = 'Active'")
# FIX: filter before join
# df.filter("status = 'Active'").join(big_table, "id")

# BAD 4: explode before filter
# df.withColumn("item", F.explode("items")).filter("item = 'X'")
# FIX: filter array first
# df.filter(F.array_contains("items","X")).withColumn("item", F.explode("items"))

# BAD 5: select * when you need 2 columns
# spark.read.parquet(path)            # reads all columns
# FIX:
# spark.read.parquet(path).select("id","salary")

print("Anti-patterns reviewed.")

## Key Takeaways

| Optimization | How to Enable |
|---|---|
| Predicate Pushdown | Automatic — filter on raw (untransformed) columns |
| Column Pruning | Automatic — `select()` only needed columns early |
| Partition Pruning | `partitionBy(col)` on write + filter on that col |
| Delta Data Skipping | Automatic with Delta — do not disable statistics |
| Z-ORDER | `OPTIMIZE ... ZORDER BY (col1, col2)` |
| Bloom Filters | `ALTER TABLE SET TBLPROPERTIES` + `OPTIMIZE` |
| ANALYZE TABLE | `ANALYZE TABLE ... COMPUTE STATISTICS FOR COLUMNS` |
| Verify all | `df.explain(mode="formatted")` — read the plan! |